# Error Analysis — Baseline (TF-IDF + Logistic Regression)

DCIT 316 — Detecting Clickbait and Sensationalism in Ghanaian Online News Headlines.

Lists every headline the class-balanced baseline gets wrong on the shared seed-42 test set, broken down by outlet. Retrains the exact Stage 3 baseline (same pipeline and split as `baseline.ipynb`).

DistilBERT and zero-shot are not included: the Kaggle runs saved only metrics and confusion matrices, not per-headline predictions. See the manuscript's Section 6 for the written interpretation.

In [1]:
import os, sys
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, os.path.abspath(".."))
from models.data import load_labeled_dataset, make_train_test_split

pd.set_option("display.max_colwidth", None)
RANDOM_SEED = 42

In [2]:
df = load_labeled_dataset("../data/processed/annotated_headlines.csv")
train_df, test_df = make_train_test_split(df, test_size=0.2, seed=RANDOM_SEED)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, class_weight="balanced")),
])
pipeline.fit(train_df["headline_text"], train_df["label"])

test_df = test_df.copy()
test_df["pred"] = pipeline.predict(test_df["headline_text"])
false_alarms = test_df[(test_df["label"] == "Informative") & (test_df["pred"] == "Clickbait")]
missed = test_df[(test_df["label"] == "Clickbait") & (test_df["pred"] == "Informative")]
print(f"False alarms (Informative -> Clickbait): {len(false_alarms)}")
print(f"Missed clickbait (Clickbait -> Informative): {len(missed)}")

False alarms (Informative -> Clickbait): 36
Missed clickbait (Clickbait -> Informative): 16


C:\Users\Y\Documents\01_Academics\L300 Sem.2\DCIT 316\end-of-sem-research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


## Errors by outlet

In [3]:
pd.DataFrame({
    "false_alarms": false_alarms["outlet"].value_counts(),
    "missed_clickbait": missed["outlet"].value_counts(),
    "clickbait_in_test": test_df[test_df["label"] == "Clickbait"]["outlet"].value_counts(),
    "headlines_in_test": test_df["outlet"].value_counts(),
}).fillna(0).astype(int).sort_values("headlines_in_test", ascending=False)

,false_alarms,missed_clickbait,clickbait_in_test,headlines_in_test
outlet,,,,
yen,13,7,10,44
zionfelix,11,3,13,32
citinewsroom,1,0,0,31
gna,0,0,1,27
ghpage,9,6,14,27
myjoyonline,2,0,1,27
graphic,0,0,0,10


## False alarms: informative headlines flagged as clickbait

In [4]:
false_alarms[["outlet", "headline_text"]].sort_values("outlet").reset_index(drop=True)

,outlet,headline_text
0,citinewsroom,Why Trade Shows & Fairs are critical for the growth of Ghana’s SMEs
1,ghpage,Ghanaian Mother Wins Hearts After Coaching Daughter To Brazilian Jiu-Jitsu Gold In The US
2,ghpage,Youngman who went swimming to impress his girlfriend dies
3,ghpage,Lady accuses her pastor of making her lose her womb
4,ghpage,Pastor jailed for scamming a teacher with the promise of getting him a job in Malaysia
5,ghpage,Police shoots and disarm juju man at Agona Amenfi
6,ghpage,GES interdicts Bole SHS teacher in viral video
7,ghpage,Wife stabs younger husband to death
8,ghpage,Ghanaians call for the immediate arrest of the Mim SHS security guard in viral video
9,ghpage,Female community members discipline husband who assaults his wife


## Missed clickbait: clickbait headlines predicted as informative

In [5]:
missed[["outlet", "headline_text"]].sort_values("outlet").reset_index(drop=True)

,outlet,headline_text
0,ghpage,Netizens call for the immediate arrest of Sexy Vanilla
1,ghpage,FBI Provides Fresh Update On Suspected $15 Million Romance Scam Case Involving Abu Trica
2,ghpage,BREAKING- Parliament passes anti-LGBTQ bill
3,ghpage,Headmaster discovered dead inside the guest house
4,ghpage,More videos from Jacobu cheating incident surface online
5,ghpage,More than KOREA War: Police face armed robbers boot for boot at GCB
6,yen,Obaa Tiwaa Shares Details of Incident That Allegedly Forced Akosua Serwaa To File for Divorce
7,yen,"Rashida Black Beauty Flaunts Her Fine Figure in a Top and Bodycon Skirt, Twirls in Front of a Camera"
8,yen,"Tracey Boakye Opens a Plush Restaurant in Trassaco, Felicia Osei and Others Attend the Launch"
9,yen,“I Dumped My Obroni Hubby After 3 Weeks in UK for Ghanaian Lover in Ghana”: Socialite Afrah Recounts


## Capitalisation is invisible to the baseline

`TfidfVectorizer` lowercases text by default, so capitalised false-urgency markers from the codebook ("BREAKING", "JUST IN") look identical to ordinary words.

In [6]:
vec = pipeline.named_steps["tfidf"]
print("lowercase:", vec.lowercase)
print(vec.build_analyzer()("BREAKING- Parliament passes anti-LGBTQ bill")[:4])

lowercase: True
['breaking', 'parliament', 'passes', 'anti']


## Summary

- **False alarms** come almost entirely from the three gossip outlets. They are mostly factual celebrity/entertainment headlines containing words like *photos*, *video* or celebrity names, plus short, dramatic crime headlines that state exactly what happened. The model reacts to topic and outlet style rather than to withheld information.
- **Missed clickbait** is concentrated in the subtle codebook cases: vague hype with nothing checkable ("Fine Figure", "Plush Restaurant", "Big And Tall"), quotes with no anchoring fact, vague teasers ("More videos … surface online"), and a capitalised "BREAKING" the model cannot see.
- Both error types point the same way as the held-out-outlet test in `baseline.ipynb`: the bag-of-words baseline learns gossip-site vocabulary, not the curiosity-gap structure that defines clickbait.